In [6]:
import os
import random
import torch
import time
import google.generativeai as genai
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

MODEL_PATH = "./final_ghost_detector" 
BASE_MODEL_ID = "distilbert-base-uncased"
GENERATIONS = 10
POPULATION_SIZE = 10
TOP_K = 3 

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running Turing Test on {device.upper()}...")

print("Loading the Judge (LoRA Classifier)...")
try:
    # Load Base
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_ID, num_labels=2
    )
    # Load Adapters
    model = PeftModel.from_pretrained(base_model, MODEL_PATH)
    model.to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
except Exception as e:
    print(f"CRITICAL ERROR loading model: {e}")
    print("Ensure 'final_ghost_detector_local' exists and contains adapter_config.json")
    exit()

Running Turing Test on CPU...
Loading the Judge (LoRA Classifier)...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1206.80it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def get_fitness_score(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)
        
        human_prob = probs[0, 0].item()
    return human_prob

gemini_model = genai.GenerativeModel('gemini-2.5-flash-lite')

def generate_initial_population(n=10):
    print(f"Genesis: Creating {n} initial organisms...")
    population = []
    # Mix of prompts to ensure diversity
    topics = ["the decay of a seaside town", "a forgotten library", "the silence of snow", "an ancient clock", "burden of family"]
    
    for _ in range(n):
        topic = random.choice(topics)
        prompt = f"Write a single, atmospheric paragraph (100 words) about {topic}."
        try:
            response = gemini_model.generate_content(prompt)
            if response.text:
                population.append(response.text.strip())
            time.sleep(0.5) # Rate limit safety
        except Exception as e:
            print(f"Gen Error: {e}")
            
    return population

In [8]:
def mutate(parent_text):
    """
    Asks Gemini to rewrite the text to sound more human, based on specific strategies.
    """
    strategies = [
        "Rewrite this to vary the sentence length significantly. Make some sentences very short and punchy.",
        "Introduce a subtle, archaic grammatical phrasing typical of the 19th century.",
        "Remove any generic transition words like 'Furthermore' or 'Moreover'.",
        "Add a specific, vivid sensory detail (smell or texture) that feels uncomfortably real.",
        "Rewrite this to sound less structured and more like a stream of consciousness."
    ]
    strategy = random.choice(strategies)
    
    prompt = f"""
    ORIGINAL TEXT:
    "{parent_text}"
    
    TASK:
    {strategy}
    Keep the core meaning but change the style to pass as a human author. 
    Output ONLY the new paragraph.
    """
    
    try:
        response = gemini_model.generate_content(prompt)
        return response.text.strip()
    except:
        return parent_text

In [9]:
def run_evolution():
    # generation
    population = generate_initial_population(POPULATION_SIZE)
    
    best_ever_score = 0.0
    best_ever_text = ""
    
    for gen in range(GENERATIONS):
        print(f"\n--- GENERATION {gen + 1} ---")
         # Respect Gemini API limits
        # testing
        scored_pop = []
        for individual in population:
            score = get_fitness_score(individual)
            scored_pop.append((score, individual))
        
        # selection
        # sort by desceding human score
        scored_pop.sort(key=lambda x: x[0], reverse=True)
        
        # Logging
        top_score = scored_pop[0][0]
        print(f"Best Human-Score: {top_score:.4f}")
        print(f"Top Text snippet: {scored_pop[0][1][:60]}...")
        
        if top_score > best_ever_score:
            best_ever_score = top_score
            best_ever_text = scored_pop[0][1]
        
        if top_score > 0.90:
            print("\nSUCCESS! The Super-Imposter has bypassed the detector (>90%).")
            break
            
        # select the top K survivors
        survivors = [x[1] for x in scored_pop[:TOP_K]]
        
        # mutation
        next_gen = []
        next_gen.extend(survivors) # keeping winners unchanged
        
        while len(next_gen) < POPULATION_SIZE:
            parent = random.choice(survivors)
            child = mutate(parent)
            next_gen.append(child)
            time.sleep(0.5)
            
        population = next_gen

    return best_ever_text, best_ever_score

In [10]:
if __name__ == "__main__":
    final_text, final_score = run_evolution()
    
    print("\n" + "="*40)
    print("EVOLUTION COMPLETE")
    print(f"Final Detector Confidence (Human): {final_score:.4f}")
    print("="*40)
    print("The Super-Imposter Paragraph:")
    print(final_text)
    
    # Save to file for your report
    with open("super_imposter.txt", "w") as f:
        f.write(f"Score: {final_score}\n\n{final_text}")  

Genesis: Creating 10 initial organisms...

--- GENERATION 1 ---
Best Human-Score: 0.0067
Top Text snippet: Salt-laced wind whispers through skeletal window frames, rat...

--- GENERATION 2 ---
Best Human-Score: 0.1839
Top Text snippet: There it was, that old grandfather clock in the dim hall, ju...

--- GENERATION 3 ---
Best Human-Score: 0.2647
Top Text snippet: Salt wind, sharp, just sighs through those empty window hole...

--- GENERATION 4 ---
Best Human-Score: 0.3585
Top Text snippet: That clock, you know, the grandfather clock in the hall, all...

--- GENERATION 5 ---
Best Human-Score: 0.3585
Top Text snippet: That clock, you know, the grandfather clock in the hall, all...

--- GENERATION 6 ---
Best Human-Score: 0.3854
Top Text snippet: That clock, you know, the hall one, the grandfather thing, a...

--- GENERATION 7 ---
Best Human-Score: 0.5970
Top Text snippet: That grandfather clock, in the hall, you understand, with it...

--- GENERATION 8 ---
Best Human-Score: 0.8006
Top Text